# Human or AI: baseline

Word TF-IDF + logistic regression. Fixed 8,000/2,000 split, seed 42. This is the starting model for the team, before tuning or adding unlabeled data.

In [1]:
from pathlib import Path
import json
import pandas as pd
from baseline import (ROOT, SEED, load_data, load_split, make_model, audit_data,
                      fit_and_evaluate, save_outputs)

DATA_PATH = ROOT / "data" / "train.csv"
df = load_data(DATA_PATH)
print("Labeled rows:", len(df))
print("Label counts:", df.label.value_counts().sort_index().to_dict())
print("Seed:", SEED)

Labeled rows: 10000
Label counts: {0: 5000, 1: 5000}
Seed: 42


## Shared split

The ID files are the team split. Existing files are loaded and checked instead of making a new split.

In [2]:
train, val = load_split(df)
split_counts = pd.DataFrame({
    "train": train.label.value_counts().sort_index(),
    "validation": val.label.value_counts().sort_index()
})
print("Train:", len(train), "Validation:", len(val))
display(split_counts)
audit = audit_data(df, train, val)
print(json.dumps(audit, indent=2))

Train: 8000 Validation: 2000
{
  "labeled_rows": 10000,
  "unique_ids": 10000,
  "missing_values": 0,
  "repeated_exact_texts": 0,
  "repeated_case_whitespace_normalized_texts": 0,
  "exact_text_overlap_between_splits": 0,
  "normalized_text_overlap_between_splits": 0,
  "near_duplicate_check": "Not performed; source/domain/generator metadata are unavailable.",
  "train_label_counts": {
    "0": 4000,
    "1": 4000
  },
  "validation_label_counts": {
    "0": 1000,
    "1": 1000
  }
}


,train,validation
label,,
0,4000,1000
1,4000,1000


## Model

Lowercase word unigrams and bigrams, min_df=2, max_features=50,000, sublinear TF and L2 normalization. No stopword removal or stemming. Logistic regression uses C=1 and liblinear. These settings were fixed before evaluation; this run does not include a parameter search.

In [3]:
settings = make_model()
print(settings)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=50000, min_df=2,
                                 ngram_range=(1, 2), sublinear_tf=True)),
                ('lr',
                 LogisticRegression(max_iter=2000, random_state=42,
                                    solver='liblinear'))])


## Fit and evaluate

Only the training partition is passed to fit, including the TF-IDF step. Fit time includes TF-IDF and logistic regression, but not file loading or metric calculation.

In [4]:
model, metrics, train_prob, val_prob = fit_and_evaluate(train, val)
summary = pd.DataFrame([{
    "Model": metrics["model"],
    "Training AUROC": metrics["training_auroc"],
    "Validation AUROC": metrics["validation_auroc"],
    "Training time (s)": metrics["training_time_seconds"]
}])
display(summary)
print("Vocabulary size:", metrics["vocabulary_size"])
print("LR iterations:", metrics["lr_iterations"])

Vocabulary size: 50000
LR iterations: 5


,Model,Training AUROC,Validation AUROC,Training time (s)
0,Word TF-IDF + Logistic Regression,0.993344,0.932705,4.653441


## Export

The prediction CSVs keep IDs, true labels and P(AI), plus a few columns for error analysis. The saved pipeline includes the fitted vectorizer.

In [5]:
save_outputs(df, train, val, model, metrics, train_prob, val_prob)
pred = pd.read_csv(ROOT / "results" / "validation_predictions.csv", dtype={"id": str})
assert len(pred) == 2000
assert set(pred.id) == set(val.id)
assert pred.predicted_ai_probability.between(0, 1).all()
display(pred.head())
print("Validation errors at threshold 0.5:")
print(pred.error_type.value_counts().to_string())

Validation errors at threshold 0.5:
error_type
correct    1734
FN          197
FP           69


,id,true_label,predicted_ai_probability,predicted_label,word_count,error_type
0,6283,0,0.330691,0,161,correct
1,2607,1,0.255880,0,179,FN
2,3980,0,0.124057,0,248,correct
3,9156,1,0.774634,1,320,correct
4,7295,0,0.215645,0,302,correct


The validation score here is for this fixed split, not a Kaggle score. The training/validation gap needs more analysis. Q1 still needs the other model comparisons and tuning; Q2 still needs the learning-curve runs. Q3-Q5 can start from this baseline and the shared split. Q4 should be updated if the selected model changes.